# Transforming data with Harmony

`earthaccess.transform()` sends data you found with `earthaccess.search_data()` to a transformation service instead of downloading the original files. The service subsets, reprojects, or reformats the data, and earthaccess downloads the (much smaller) results.

Transformations use a plugin system. The first backend is [NASA Harmony](https://harmony.earthdata.nasa.gov), built on [harmony-py](https://github.com/nasa/harmony-py). For each collection, earthaccess:

1. queries Harmony's **capabilities** for the collection and checks that the requested transformation is supported,
2. builds and **submits** a Harmony request,
3. **waits** for the job to complete, and
4. **downloads** the job outputs.

## Requirements

```bash
pip install "earthaccess[harmony]" xarray h5netcdf dask matplotlib
```

You need an [Earthdata Login](https://urs.earthdata.nasa.gov) account. This example uses the GHRSST MUR sea surface temperature dataset (`MUR-JPL-L4-GLOB-v4.1`) from PO.DAAC. Each original granule is ~400 MB; the subset we request is a few hundred KB.

In [ ]:
import json

import earthaccess

earthaccess.__version__, earthaccess.transformations.available_backends()

## 1. Log in

The Harmony backend reuses your earthaccess credentials (EDL token), and targets the Harmony environment that matches the system you log in to (`earthaccess.PROD` → Harmony production, `earthaccess.UAT` → Harmony UAT).

In [ ]:
auth = earthaccess.login()

## 2. Search

This is exactly the same search you would do before `earthaccess.download()`. We look for two days of MUR SST over the Gulf of Mexico.

In [ ]:
bounding_box = (-98.0, 18.0, -80.0, 31.0)  # west, south, east, north
temporal = ("2024-08-01", "2024-08-02")

granules = earthaccess.search_data(
    short_name="MUR-JPL-L4-GLOB-v4.1",
    bounding_box=bounding_box,
    temporal=temporal,
)
print(f"{len(granules)} granules, {sum(g.size for g in granules):.0f} MB in total")
granules[0]

## 3. Inspect the collection's transformation capabilities (optional)

`transform()` checks capabilities automatically, but you can look at them yourself to see which operations, output formats, and variables Harmony supports for the collection.

In [ ]:
capabilities = earthaccess.transformation_capabilities(granules[0])

summary = {
    key: capabilities.get(key)
    for key in [
        "conceptId",
        "shortName",
        "variableSubset",
        "bboxSubset",
        "temporalSubset",
        "shapeSubset",
        "concatenate",
        "reproject",
        "outputFormats",
    ]
}
summary["services"] = [s["name"] for s in capabilities.get("services", [])]
summary["variables"] = [v["name"] for v in capabilities.get("variables", [])]
print(json.dumps(summary, indent=2))

## 4. Transform and download

Instead of `earthaccess.download(granules)`, call `earthaccess.transform(granules, ...)`. Here we subset to a single variable and our bounding box. This submits one Harmony job (one per collection), shows its progress, and downloads the outputs.

Additional keyword arguments are passed through to `harmony.Request`.

In [ ]:
results = earthaccess.transform(
    granules,
    variables=["analysed_sst"],
    bounding_box=bounding_box,
    local_path="../../../notebooks/data/harmony-demo",
)

for result in results:
    print(f"Job {result.job_id}: {result.status} - {result.message}")
    print(f"  https://harmony.earthdata.nasa.gov/jobs/{result.job_id}")
    for path in result.files:
        print(f"  {path} ({path.stat().st_size / 1024:.0f} KB)")

## 5. Look at the results

In [ ]:
import xarray as xr

files = sorted(f for r in results for f in r.files)
ds = xr.open_mfdataset(files, combine="by_coords")
ds

In [ ]:
ds["analysed_sst"].isel(time=0).plot(figsize=(10, 6), cmap="RdYlBu_r");

## Getting URLs without downloading

Pass `download=False` to only run the job and get output URLs, e.g. to stream them later or to request S3 links when running in AWS `us-west-2` (`backend_options={"link_type": "s3"}`).

In [ ]:
(result,) = earthaccess.transform(
    granules[:1],
    variables=["analysed_sst"],
    bounding_box=bounding_box,
    download=False,
)
result.urls

## Unsupported requests fail early

Before submitting, `transform()` checks the request against the collection's capabilities, and raises `TransformationNotSupported` without creating a job if Harmony can't fulfill it. To show this without any risk of submitting a job, we run that same check directly against the capabilities we fetched above, asking for a variable that doesn't exist and an output format that isn't offered.

In [ ]:
from earthaccess.exceptions import TransformationNotSupported
from earthaccess.transformations import TransformationRequest, get_backend

harmony_backend = get_backend("harmony")(auth)
request = TransformationRequest(
    collection_id=capabilities["conceptId"],
    variables=["not_a_variable"],
    output_format="image/not-a-format",
)

try:
    harmony_backend.validate(request, capabilities)
except TransformationNotSupported as err:
    print(err)
else:
    print("Harmony advertises no restrictions that rule this request out.")

## Writing your own backend

Other transformation services can be plugged in by subclassing `earthaccess.transformations.TransformationBackend` and implementing `capabilities`, `submit`, `wait`, `result_urls`, and `download` (and optionally `validate`). The shared `run()` method drives the capabilities → submit → wait → download workflow.

Register a backend at runtime with `earthaccess.transformations.register_backend(MyBackend)`, or from a package via an entry point:

```toml
[project.entry-points."earthaccess.transformations"]
mybackend = "my_package.module:MyBackend"
```

and then use it with `earthaccess.transform(granules, backend="mybackend", ...)`.